# Mini Neural Search — V3: Train, Evaluate, and Search

This notebook is self-contained and uses only `data/v3/`. It does not modify the existing V2 notebook or checkpoints.

**Run the cells from top to bottom.** It trains a small custom PyTorch neural retrieval model from scratch, evaluates it on V3 validation data, saves the best weights under the current project's `checkpoints/` directory, evaluates the test set once, and lets you search with a new query.

Expected V3 files:
- `data/v3/documents.json`
- `data/v3/train.json`
- `data/v3/validation.json`
- `data/v3/test.json`

The notebook discovers the project root by searching the current directory and its parents for `data/v3/documents.json`. If it cannot find the files, fix `PROJECT_ROOT` in Cell 2.

This is a learning baseline, not a promise of a particular retrieval score. We will use validation metrics to assess it before changing the model.

In [1]:
# Cell 1 — Imports and reproducibility

import json
import random
import re
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("PyTorch:", torch.__version__)
print("Device:", device)

PyTorch: 2.14.0
Device: cpu


In [2]:
# Cell 2 — Locate this project's V3 data and checkpoint directory

# Works if the notebook is opened from the project root or a subdirectory.
PROJECT_ROOT = None
here = Path.cwd().resolve()
for candidate in [here, *here.parents]:
    if (candidate / "data" / "v3" / "documents.json").exists():
        PROJECT_ROOT = candidate
        break

# If auto-detection fails, uncomment and set this to the root of the project
# currently open in Cursor:
# PROJECT_ROOT = Path("/absolute/path/to/MINI-NEURAL-SEARCH").resolve()

if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "Could not find data/v3/documents.json from the notebook's current directory. "
        "Set PROJECT_ROOT to the project root in this cell."
    )

DATA_DIR = PROJECT_ROOT / "data" / "v3"
CHECKPOINT_DIR = PROJECT_ROOT / "checkpoints"
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("V3 data directory:", DATA_DIR)
print("Checkpoint directory:", CHECKPOINT_DIR)
print("V3 files:", sorted(p.name for p in DATA_DIR.iterdir()))

Project root: /Users/sushilpratapsingh/projects/mini-neural-search
V3 data directory: /Users/sushilpratapsingh/projects/mini-neural-search/data/v3
Checkpoint directory: /Users/sushilpratapsingh/projects/mini-neural-search/checkpoints
V3 files: ['README.md', 'documents.json', 'manifest.json', 'test.json', 'train.json', 'validate_dataset.py', 'validation.json']


In [3]:
# Cell 3 — Load the V3 documents and query pairs

def load_json(path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

documents = load_json(DATA_DIR / "documents.json")
train_pairs = load_json(DATA_DIR / "train.json")
validation_pairs = load_json(DATA_DIR / "validation.json")
test_pairs = load_json(DATA_DIR / "test.json")

# Accept either a raw JSON list or a wrapper object with a common collection key.
def unwrap_records(data, possible_keys):
    if isinstance(data, list):
        return data
    if isinstance(data, dict):
        for key in possible_keys:
            if key in data and isinstance(data[key], list):
                return data[key]
    raise ValueError(f"Expected a list of records or one of these keys: {possible_keys}")

documents = unwrap_records(documents, ["documents", "data", "items"])
train_pairs = unwrap_records(train_pairs, ["pairs", "queries", "data", "items", "train"])
validation_pairs = unwrap_records(validation_pairs, ["pairs", "queries", "data", "items", "validation"])
test_pairs = unwrap_records(test_pairs, ["pairs", "queries", "data", "items", "test"])

required_doc_keys = {"id", "title", "text"}
required_pair_keys = {"query", "document_id"}
assert documents and required_doc_keys.issubset(documents[0]), f"Document needs keys {required_doc_keys}"
assert train_pairs and required_pair_keys.issubset(train_pairs[0]), f"Query pair needs keys {required_pair_keys}"

document_lookup = {str(d["id"]): d for d in documents}
document_ids = list(document_lookup.keys())

for split_name, pairs in [("train", train_pairs), ("validation", validation_pairs), ("test", test_pairs)]:
    missing = sorted({str(p["document_id"]) for p in pairs} - set(document_lookup))
    if missing:
        raise ValueError(f"{split_name} contains document IDs absent from documents.json: {missing[:10]}")
    blank = [i for i, p in enumerate(pairs) if not str(p["query"]).strip()]
    if blank:
        raise ValueError(f"{split_name} contains blank queries at rows {blank[:10]}")

print(f"Documents: {len(documents)}")
print(f"Training pairs: {len(train_pairs)}")
print(f"Validation pairs: {len(validation_pairs)}")
print(f"Test pairs: {len(test_pairs)}")
display(pd.DataFrame(documents).head(3))
display(pd.DataFrame(train_pairs).head(5))

Documents: 100
Training pairs: 800
Validation pairs: 200
Test pairs: 200


,id,category,title,text,keywords,aliases
0,java-001,java,Collections Framework,Java's Collections Framework provides interfac...,"[ArrayList, LinkedList, HashSet, TreeSet, Hash...",[]
1,java-002,java,HashMap and hashing,HashMap stores key-value mappings in hash buck...,"[hashCode, equals, buckets, collisions, resizi...",[]
2,java-003,java,Exception handling,Java exception handling separates normal contr...,"[try catch finally, checked exceptions, unchec...",[]


,query,document_id
0,Give me an engineering explanation of collecti...,java-001
1,What happens internally with Java List Set Map...,java-001
2,I need to choose an approach for ArrayList ver...,java-001
3,What can go wrong when using collection iterat...,java-001
4,How can I diagnose a failure involving collect...,java-001


In [4]:
# Cell 4 — Validate the dataset before training

print("Unique document IDs:", len(set(str(d["id"]) for d in documents)))
print("Duplicate document IDs:", len(documents) - len(set(str(d["id"]) for d in documents)))

for name, pairs in [("train", train_pairs), ("validation", validation_pairs), ("test", test_pairs)]:
    counts = Counter(str(p["document_id"]) for p in pairs)
    print(f"{name}: {len(pairs)} pairs, {len(counts)} target documents, "
          f"{sum(v > 1 for v in counts.values())} documents with multiple queries")

# Query overlap check (exact text after lowercasing and whitespace normalization).
def normalize_query(q):
    return " ".join(str(q).lower().split())

train_queries = {normalize_query(p["query"]) for p in train_pairs}
val_queries = {normalize_query(p["query"]) for p in validation_pairs}
test_queries = {normalize_query(p["query"]) for p in test_pairs}
print("Exact train/validation query overlap:", len(train_queries & val_queries))
print("Exact train/test query overlap:", len(train_queries & test_queries))
print("Exact validation/test query overlap:", len(val_queries & test_queries))

# Preview documents and their actual specificity.
display(pd.DataFrame([
    {"id": d["id"], "title": d["title"], "category": d.get("category", ""), "text_chars": len(d["text"])}
    for d in documents
]).head(10))

Unique document IDs: 100
Duplicate document IDs: 0
train: 800 pairs, 100 target documents, 100 documents with multiple queries
validation: 200 pairs, 100 target documents, 100 documents with multiple queries
test: 200 pairs, 100 target documents, 100 documents with multiple queries
Exact train/validation query overlap: 0
Exact train/test query overlap: 0
Exact validation/test query overlap: 0


,id,title,category,text_chars
0,java-001,Collections Framework,java,859
1,java-002,HashMap and hashing,java,736
2,java-003,Exception handling,java,748
3,java-004,Multithreading,java,758
4,java-005,Synchronization and locks,java,733
5,java-006,Garbage collection,java,786
6,java-007,JVM memory areas,java,743
7,java-008,Streams,java,706
8,java-009,CompletableFuture,java,709
9,java-010,Records and immutability,java,643


In [5]:
# Cell 5 — Tokenization and vocabulary (built from training-available text only)

# Lowercase and separate punctuation so e.g. "HashMap," and "HashMap" share a token.
TOKEN_PATTERN = re.compile(r"[a-z0-9]+(?:[.+#/_-][a-z0-9]+)*|[^\w\s]", re.IGNORECASE)

def tokenize(text):
    return TOKEN_PATTERN.findall(str(text).lower())

def document_text(doc):
    # Title is included consistently in training, validation, and search.
    return f"{doc.get('title', '')}. {doc.get('text', '')}"

# Do not use validation/test queries to build the vocabulary.
vocab_counter = Counter()
for doc in documents:
    vocab_counter.update(tokenize(document_text(doc)))
for pair in train_pairs:
    vocab_counter.update(tokenize(pair["query"]))

PAD_TOKEN = "<pad>"
UNK_TOKEN = "<unk>"
vocab = {PAD_TOKEN: 0, UNK_TOKEN: 1}
for token, _ in vocab_counter.most_common():
    if token not in vocab:
        vocab[token] = len(vocab)

PAD_ID = vocab[PAD_TOKEN]
UNK_ID = vocab[UNK_TOKEN]

def encode_text(text):
    tokens = tokenize(text)
    ids = [vocab.get(token, UNK_ID) for token in tokens]
    return ids if ids else [UNK_ID]

print("Vocabulary size:", len(vocab))
print("Example tokens:", tokenize("How do I diagnose Java HashMap performance?"))
print("Unknown-token rate in validation queries:",
      round(sum(t not in vocab for p in validation_pairs for t in tokenize(p["query"])) /
            max(1, sum(len(tokenize(p["query"])) for p in validation_pairs)), 4))

Vocabulary size: 2425
Example tokens: ['how', 'do', 'i', 'diagnose', 'java', 'hashmap', 'performance', '?']
Unknown-token rate in validation queries: 0.1498


In [6]:
# Cell 6 — Convert text to padded tensors

MAX_LENGTH = 160

def texts_to_padded_tensor(texts, max_length=MAX_LENGTH):
    encoded = [encode_text(t)[:max_length] for t in texts]
    width = max(1, max(len(x) for x in encoded))
    batch = torch.full((len(encoded), width), PAD_ID, dtype=torch.long)
    for i, ids in enumerate(encoded):
        batch[i, :len(ids)] = torch.tensor(ids, dtype=torch.long)
    return batch

# Sanity checks
example_batch = texts_to_padded_tensor([
    document_text(documents[0]),
    "How do I troubleshoot a production issue?"
])
print("Example tensor shape:", tuple(example_batch.shape))
print("PAD ID:", PAD_ID, "| UNK ID:", UNK_ID)

Example tensor shape: (2, 136)
PAD ID: 0 | UNK ID: 1


In [7]:
# Cell 7 — Define the custom neural text encoder

class TextEncoder(nn.Module):
    def __init__(self, vocab_size, embedding_dim=64, output_dim=32, pad_id=0):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=pad_id)
        self.projection = nn.Linear(embedding_dim, output_dim)

    def forward(self, token_ids):
        mask = (token_ids != PAD_ID).unsqueeze(-1).to(torch.float32)
        word_vectors = self.embedding(token_ids)
        summed = (word_vectors * mask).sum(dim=1)
        lengths = mask.sum(dim=1).clamp(min=1.0)
        pooled = summed / lengths
        projected = self.projection(pooled)
        return F.normalize(projected, p=2, dim=1)

EMBEDDING_DIM = 64
OUTPUT_DIM = 32
model = TextEncoder(
    vocab_size=len(vocab),
    embedding_dim=EMBEDDING_DIM,
    output_dim=OUTPUT_DIM,
    pad_id=PAD_ID
).to(device)

parameter_count = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(model)
print("Trainable parameters:", parameter_count)

TextEncoder(
  (embedding): Embedding(2425, 64, padding_idx=0)
  (projection): Linear(in_features=64, out_features=32, bias=True)
)
Trainable parameters: 157280


In [8]:
# Cell 8 — Dataset and training settings

class QueryDocumentDataset(Dataset):
    def __init__(self, pairs):
        self.pairs = pairs

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, index):
        pair = self.pairs[index]
        doc_id = str(pair["document_id"])
        return str(pair["query"]), doc_id

BATCH_SIZE = 64
EPOCHS = 40
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-5
TEMPERATURE = 0.08

train_loader = DataLoader(
    QueryDocumentDataset(train_pairs),
    batch_size=BATCH_SIZE,
    shuffle=True,
    drop_last=False
)

print("Training batches per epoch:", len(train_loader))
print("Epochs:", EPOCHS, "| Batch size:", BATCH_SIZE, "| Temperature:", TEMPERATURE)

Training batches per epoch: 13
Epochs: 40 | Batch size: 64 | Temperature: 0.08


In [9]:
# Cell 9 — Encode all candidate documents and compute retrieval metrics

@torch.no_grad()
def encode_documents(model):
    model.eval()
    texts = [document_text(document_lookup[doc_id]) for doc_id in document_ids]
    vectors = []
    for start in range(0, len(texts), 128):
        token_batch = texts_to_padded_tensor(texts[start:start + 128]).to(device)
        vectors.append(model(token_batch))
    return torch.cat(vectors, dim=0)

@torch.no_grad()
def evaluate_retrieval(model, pairs, report=False):
    model.eval()
    doc_vectors = encode_documents(model)
    query_texts = [str(p["query"]) for p in pairs]
    query_vectors = []
    for start in range(0, len(query_texts), 128):
        token_batch = texts_to_padded_tensor(query_texts[start:start + 128]).to(device)
        query_vectors.append(model(token_batch))
    query_vectors = torch.cat(query_vectors, dim=0)

    similarity = query_vectors @ doc_vectors.T
    id_to_index = {doc_id: i for i, doc_id in enumerate(document_ids)}
    ranks = []
    top_rows = []
    for i, pair in enumerate(pairs):
        expected_id = str(pair["document_id"])
        order = torch.argsort(similarity[i], descending=True)
        ranked_ids = [document_ids[j] for j in order.tolist()]
        rank = ranked_ids.index(expected_id) + 1
        ranks.append(rank)
        top5_idx = order[:5].tolist()
        top_rows.append({
            "query": pair["query"],
            "expected_id": expected_id,
            "expected_rank": rank,
            "top5_ids": [document_ids[j] for j in top5_idx],
            "top5_titles": [document_lookup[document_ids[j]]["title"] for j in top5_idx],
            "top5_scores": [round(float(similarity[i, j].item()), 4) for j in top5_idx],
        })

    ranks_arr = np.asarray(ranks)
    metrics = {
        "queries": len(pairs),
        "Recall@1": float(np.mean(ranks_arr <= 1)),
        "Recall@3": float(np.mean(ranks_arr <= 3)),
        "Recall@5": float(np.mean(ranks_arr <= 5)),
        "MRR": float(np.mean(1.0 / ranks_arr)),
    }
    return (metrics, pd.DataFrame(top_rows)) if report else metrics

print("Initial validation:", evaluate_retrieval(model, validation_pairs))

Initial validation: {'queries': 200, 'Recall@1': 0.03, 'Recall@3': 0.085, 'Recall@5': 0.12, 'MRR': 0.09973947439992635}


In [10]:
# Cell 10 — Train with in-batch contrastive softmax loss

# Documents are encoded once per unique target within a batch.
# If multiple queries in the same batch target the same document, they are
# treated as positives rather than incorrectly treating that document as a negative.

optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
best_mrr = -1.0
best_epoch = None
history = []
BEST_CHECKPOINT_PATH = CHECKPOINT_DIR / "v3_best_model.pth"

for epoch in range(1, EPOCHS + 1):
    model.train()
    total_loss = 0.0
    seen = 0

    for queries_batch, target_ids_batch in train_loader:
        queries_batch = list(queries_batch)
        target_ids_batch = [str(x) for x in target_ids_batch]

        # Deduplicate target documents in the batch and keep a stable index.
        unique_target_ids = list(dict.fromkeys(target_ids_batch))
        doc_texts_batch = [document_text(document_lookup[doc_id]) for doc_id in unique_target_ids]

        query_tokens = texts_to_padded_tensor(queries_batch).to(device)
        doc_tokens = texts_to_padded_tensor(doc_texts_batch).to(device)

        query_vectors = model(query_tokens)
        doc_vectors = model(doc_tokens)

        optimizer.zero_grad(set_to_none=True)
        logits = (query_vectors @ doc_vectors.T) / TEMPERATURE
        target_positions = torch.tensor(
            [unique_target_ids.index(doc_id) for doc_id in target_ids_batch],
            dtype=torch.long,
            device=device
        )
        # Each query's target is the matching document column. Candidate documents
        # are deduplicated, so two queries for the same document share one target column.
        loss = F.cross_entropy(logits, target_positions)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        total_loss += float(loss.item()) * len(queries_batch)
        seen += len(queries_batch)

    train_loss = total_loss / max(seen, 1)
    val_metrics = evaluate_retrieval(model, validation_pairs)
    row = {"epoch": epoch, "train_loss": train_loss, **val_metrics}
    history.append(row)

    if val_metrics["MRR"] > best_mrr:
        best_mrr = val_metrics["MRR"]
        best_epoch = epoch
        torch.save({
            "model_state_dict": model.state_dict(),
            "epoch": epoch,
            "validation_metrics": val_metrics,
            "vocab": vocab,
            "config": {
                "embedding_dim": EMBEDDING_DIM,
                "output_dim": OUTPUT_DIM,
                "max_length": MAX_LENGTH,
                "temperature": TEMPERATURE,
            },
        }, BEST_CHECKPOINT_PATH)

    if epoch == 1 or epoch % 5 == 0 or epoch == EPOCHS:
        print(
            f"Epoch {epoch:02d}/{EPOCHS} | loss={train_loss:.4f} | "
            f"R@1={val_metrics['Recall@1']:.3f} | "
            f"R@5={val_metrics['Recall@5']:.3f} | MRR={val_metrics['MRR']:.4f}"
        )

print("\nBest epoch:", best_epoch)
print("Best validation MRR:", round(best_mrr, 4))
print("Saved checkpoint:", BEST_CHECKPOINT_PATH.resolve())

Epoch 01/40 | loss=3.8855 | R@1=0.050 | R@5=0.150 | MRR=0.1213
Epoch 05/40 | loss=2.4730 | R@1=0.090 | R@5=0.220 | MRR=0.1834
Epoch 10/40 | loss=1.2058 | R@1=0.100 | R@5=0.290 | MRR=0.2039
Epoch 15/40 | loss=0.5950 | R@1=0.100 | R@5=0.305 | MRR=0.2129
Epoch 20/40 | loss=0.3246 | R@1=0.100 | R@5=0.330 | MRR=0.2214
Epoch 25/40 | loss=0.1721 | R@1=0.105 | R@5=0.340 | MRR=0.2292
Epoch 30/40 | loss=0.1172 | R@1=0.115 | R@5=0.345 | MRR=0.2390
Epoch 35/40 | loss=0.0821 | R@1=0.100 | R@5=0.365 | MRR=0.2375
Epoch 40/40 | loss=0.0594 | R@1=0.105 | R@5=0.380 | MRR=0.2436

Best epoch: 40
Best validation MRR: 0.2436
Saved checkpoint: /Users/sushilpratapsingh/projects/mini-neural-search/checkpoints/v3_best_model.pth


In [11]:
# Cell 11 — Review training history and load the best validation checkpoint

history_df = pd.DataFrame(history)
display(history_df.sort_values("MRR", ascending=False).head(10))

checkpoint = torch.load(BEST_CHECKPOINT_PATH, map_location=device, weights_only=False)
model.load_state_dict(checkpoint["model_state_dict"])
model.eval()

print("Loaded best epoch:", checkpoint["epoch"])
print("Saved validation metrics:", checkpoint["validation_metrics"])
print("Checkpoint exists:", BEST_CHECKPOINT_PATH.exists())
print("Checkpoint size (bytes):", BEST_CHECKPOINT_PATH.stat().st_size)

,epoch,train_loss,queries,Recall@1,Recall@3,Recall@5,MRR
39,40,0.059425,200,0.105,0.245,0.380,0.243623
28,29,0.129099,200,0.120,0.245,0.340,0.242656
30,31,0.106528,200,0.115,0.230,0.360,0.242468
35,36,0.076600,200,0.105,0.250,0.370,0.240160
38,39,0.062188,200,0.100,0.240,0.380,0.239785
29,30,0.117183,200,0.115,0.235,0.345,0.238956
31,32,0.094410,200,0.110,0.230,0.340,0.238525
32,33,0.096116,200,0.110,0.235,0.345,0.237942
33,34,0.080824,200,0.105,0.240,0.355,0.237903
34,35,0.082092,200,0.100,0.245,0.365,0.237488


Loaded best epoch: 40
Saved validation metrics: {'queries': 200, 'Recall@1': 0.105, 'Recall@3': 0.245, 'Recall@5': 0.38, 'MRR': 0.24362294818150132}
Checkpoint exists: True
Checkpoint size (bytes): 680919


In [12]:
# Cell 12 — Final test-set evaluation (run once after model selection)

test_metrics, test_results_df = evaluate_retrieval(model, test_pairs, report=True)
print("V3 test metrics")
for metric, value in test_metrics.items():
    print(f"{metric}: {value:.4f}" if isinstance(value, float) else f"{metric}: {value}")

display(test_results_df.head(10))

V3 test metrics
queries: 200
Recall@1: 0.3850
Recall@3: 0.5850
Recall@5: 0.7100
MRR: 0.5307


,query,expected_id,expected_rank,top5_ids,top5_titles,top5_scores
0,How would you investigate and fix a real incid...,java-001,5,"[messaging-005, performance-008, microservices...","[Kafka message keys, Response compression, Sag...","[0.4922, 0.4835, 0.4465, 0.4449, 0.4393]"
1,"What are the important limits, edge cases, and...",java-001,3,"[performance-008, observability-010, java-001,...","[Response compression, Incident dashboards, Co...","[0.4364, 0.422, 0.3863, 0.3627, 0.3536]"
2,How would you investigate and fix a real incid...,java-002,19,"[observability-007, java-003, spring-005, spri...","[SLI SLO and error budgets, Exception handling...","[0.3979, 0.3662, 0.3655, 0.343, 0.3153]"
3,"What are the important limits, edge cases, and...",java-002,23,"[observability-010, security-003, database-001...","[Incident dashboards, JWT tokens, Database ind...","[0.423, 0.3856, 0.3367, 0.3233, 0.3046]"
4,How would you investigate and fix a real incid...,java-003,1,"[java-003, performance-008, observability-010,...","[Exception handling, Response compression, Inc...","[0.5898, 0.5052, 0.4648, 0.3839, 0.3798]"
5,"What are the important limits, edge cases, and...",java-003,1,"[java-003, database-001, spring-004, performan...","[Exception handling, Database indexes, Actuato...","[0.4892, 0.462, 0.4593, 0.4219, 0.411]"
6,How would you investigate and fix a real incid...,java-004,7,"[java-003, messaging-006, messaging-005, obser...","[Exception handling, Dead letter queues, Kafka...","[0.4198, 0.3409, 0.3356, 0.3339, 0.3062]"
7,"What are the important limits, edge cases, and...",java-004,10,"[observability-010, database-001, security-002...","[Incident dashboards, Database indexes, OpenID...","[0.4503, 0.3437, 0.2987, 0.2813, 0.2727]"
8,How would you investigate and fix a real incid...,java-005,2,"[messaging-005, java-005, performance-005, mes...","[Kafka message keys, Synchronization and locks...","[0.4318, 0.4023, 0.3941, 0.335, 0.3107]"
9,"What are the important limits, edge cases, and...",java-005,17,"[observability-010, database-001, messaging-01...","[Incident dashboards, Database indexes, Asynch...","[0.4815, 0.4411, 0.3336, 0.3264, 0.2604]"


In [13]:
# Cell 13 — Inspect the hardest validation queries

val_metrics, validation_results_df = evaluate_retrieval(model, validation_pairs, report=True)
errors_df = validation_results_df[validation_results_df["expected_rank"] > 1].copy()
errors_df = errors_df.sort_values("expected_rank", ascending=False)

print("Validation metrics:", val_metrics)
print("Incorrect top-1 predictions:", int((validation_results_df["expected_rank"] > 1).sum()))
display(errors_df[["query", "expected_id", "expected_rank", "top5_ids", "top5_titles", "top5_scores"]].head(20))

Validation metrics: {'queries': 200, 'Recall@1': 0.105, 'Recall@3': 0.245, 'Recall@5': 0.38, 'MRR': 0.24362294818150132}
Incorrect top-1 predictions: 179


,query,expected_id,expected_rank,top5_ids,top5_titles,top5_scores
98,A production system is behaving unexpectedly a...,database-010,82,"[observability-005, aws-004, database-009, con...","[Health checks, ECS Fargate containers, Optimi...","[0.5149, 0.4857, 0.4529, 0.4193, 0.4042]"
140,A production system is behaving unexpectedly a...,messaging-001,78,"[aws-002, containers-001, observability-005, m...","[S3 object storage, Docker images, Health chec...","[0.4858, 0.416, 0.4143, 0.4105, 0.3962]"
35,Which practices make spring security reliable ...,spring-008,71,"[containers-005, messaging-003, containers-009...","[Kubernetes Deployments, Kafka offsets, Readin...","[0.5042, 0.4524, 0.4228, 0.4115, 0.4007]"
130,A production system is behaving unexpectedly a...,security-006,66,"[observability-005, containers-001, database-0...","[Health checks, Docker images, Optimistic conc...","[0.6885, 0.5801, 0.5102, 0.4234, 0.3696]"
162,A production system is behaving unexpectedly a...,observability-002,65,"[containers-005, observability-005, microservi...","[Kubernetes Deployments, Health checks, Saga d...","[0.4666, 0.4658, 0.4011, 0.3488, 0.3407]"
21,Which practices make dependency injection reli...,spring-001,65,"[messaging-003, containers-005, containers-009...","[Kafka offsets, Kubernetes Deployments, Readin...","[0.609, 0.5477, 0.5051, 0.3842, 0.3593]"
58,A production system is behaving unexpectedly a...,microservices-010,64,"[observability-005, containers-001, database-0...","[Health checks, Docker images, Optimistic conc...","[0.6269, 0.595, 0.4347, 0.3832, 0.3658]"
5,Which practices make exception handling reliab...,java-003,63,"[messaging-003, containers-005, containers-009...","[Kafka offsets, Kubernetes Deployments, Readin...","[0.6075, 0.5447, 0.5163, 0.4007, 0.3959]"
3,Which practices make hashmap and hashing relia...,java-002,63,"[messaging-003, aws-006, spring-003, container...","[Kafka offsets, Application Load Balancer, Ext...","[0.6093, 0.5399, 0.5173, 0.4998, 0.4804]"
15,Which practices make streams reliable and main...,java-008,62,"[messaging-003, containers-005, observability-...","[Kafka offsets, Kubernetes Deployments, Resour...","[0.6481, 0.5745, 0.46, 0.4263, 0.4199]"


In [ ]:
# Cell 14 — Search using a brand-new query

def search(query, top_k=5):
    query = str(query).strip()
    if not query:
        raise ValueError("Please enter a non-empty search query.")
    if not 1 <= top_k <= len(documents):
        raise ValueError(f"top_k must be between 1 and {len(documents)}")

    model.eval()
    with torch.no_grad():
        query_tokens = texts_to_padded_tensor([query]).to(device)
        query_vector = model(query_tokens)
        doc_vectors = encode_documents(model)
        scores = (query_vector @ doc_vectors.T).squeeze(0)
        top_indices = torch.argsort(scores, descending=True)[:top_k].tolist()

    results = []
    for rank, idx in enumerate(top_indices, start=1):
        doc_id = document_ids[idx]
        doc = document_lookup[doc_id]
        results.append({
            "rank": rank,
            "document_id": doc_id,
            "title": doc.get("title", ""),
            "category": doc.get("category", ""),
            "score": round(float(scores[idx].item()), 4),
            "text_preview": str(doc.get("text", ""))[:300],
        })
    return pd.DataFrame(results)

# Replace this with any query you have not used for training.
# new_query = "How can I investigate deadlocks and lock contention in a production Java service?"
# display(search(new_query, top_k=5))


# new_query = "How do Java synchronized blocks and locks prevent race conditions?"
# display(search(new_query, top_k=5))

# new_query = "How do I troubleshoot slow SQL queries and database locks?"
# display(search(new_query, top_k=5))


new_query = "How can I diagnose high CPU and memory usage in a production service?"
display(search(new_query, top_k=5))

,rank,document_id,title,category,score,text_preview
0,1,containers-005,Kubernetes Deployments,containers,0.5091,A Kubernetes Deployment manages ReplicaSets an...
1,2,microservices-004,Saga distributed transactions,microservices,0.4752,A Saga coordinates a business transaction acro...
2,3,java-005,Synchronization and locks,java,0.4586,Synchronization protects invariants when multi...
3,4,messaging-003,Kafka offsets,messaging,0.4551,Kafka offsets identify positions in a partitio...
4,5,containers-007,ConfigMaps and Secrets,containers,0.4149,Kubernetes ConfigMaps hold non-sensitive confi...


## How to use this notebook

1. Run all cells in order.
2. Confirm Cell 2 points to the `MINI-NEURAL-SEARCH` project root and `data/v3`.
3. Check Cell 3 counts and Cell 4 overlap results before training.
4. Wait for Cell 10 to finish. It writes `checkpoints/v3_best_model.pth` in the detected project root.
5. Review validation metrics and the test metrics. Do not use test results to tune the model.
6. Change `new_query` in Cell 14 to test a query that is not in the training file.

If scores are poor, do not immediately change the architecture. First inspect Cell 13's failure examples and verify that documents have distinct technical content and the expected document IDs are correct. The model, tokenizer, vocabulary, and checkpoint in this notebook are a matched set; use the checkpoint produced by this notebook with this notebook's vocabulary/model configuration.